# Quick Maths with Matrices!
---

<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Quick-Maths-with-Matrices!" data-toc-modified-id="Quick-Maths-with-Matrices!-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Quick Maths with Matrices!</a></span><ul class="toc-item"><li><span><a href="#Import-Libraries" data-toc-modified-id="Import-Libraries-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Import Libraries</a></span></li><li><span><a href="#Test-Framework" data-toc-modified-id="Test-Framework-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Test Framework</a></span></li></ul></li><li><span><a href="#Optimizing-Matrix-Multiplications" data-toc-modified-id="Optimizing-Matrix-Multiplications-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Optimizing Matrix Multiplications</a></span><ul class="toc-item"><li><span><a href="#For-Loop" data-toc-modified-id="For-Loop-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>For Loop</a></span></li><li><span><a href="#Array-Slicing" data-toc-modified-id="Array-Slicing-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>Array Slicing</a></span></li><li><span><a href="#Improvement-with-array-slicing" data-toc-modified-id="Improvement-with-array-slicing-2.3"><span class="toc-item-num">2.3&nbsp;&nbsp;</span>Improvement with array slicing</a></span></li><li><span><a href="#Array-Broadcasting" data-toc-modified-id="Array-Broadcasting-2.4"><span class="toc-item-num">2.4&nbsp;&nbsp;</span>Array Broadcasting</a></span></li><li><span><a href="#Improvement-with-array-broadcasting" data-toc-modified-id="Improvement-with-array-broadcasting-2.5"><span class="toc-item-num">2.5&nbsp;&nbsp;</span>Improvement with array broadcasting</a></span></li><li><span><a href="#Einstein-Sum" data-toc-modified-id="Einstein-Sum-2.6"><span class="toc-item-num">2.6&nbsp;&nbsp;</span>Einstein Sum</a></span></li><li><span><a href="#Improvement-with-einstein-sum" data-toc-modified-id="Improvement-with-einstein-sum-2.7"><span class="toc-item-num">2.7&nbsp;&nbsp;</span>Improvement with einstein sum</a></span></li><li><span><a href="#Linear-Algebra-Libraries" data-toc-modified-id="Linear-Algebra-Libraries-2.8"><span class="toc-item-num">2.8&nbsp;&nbsp;</span>Linear Algebra Libraries</a></span></li><li><span><a href="#Improvement-with-linear-algebra-libraries" data-toc-modified-id="Improvement-with-linear-algebra-libraries-2.9"><span class="toc-item-num">2.9&nbsp;&nbsp;</span>Improvement with linear algebra libraries</a></span></li></ul></li></ul></div>

## Import Libraries

In [13]:
import torch
import timeit
import operator
from functools import partial

## Test Framework

In [14]:
def test(a, b, compare, compare_name=None):
    if compare_name is None:
        compare_name = compare.__name__
    assert compare(a, b),\
    f"{compare_name} check failed:\n{a}\n{b}"

def test_equality(a, b):
    test(a, b, operator.eq, "Equality")

def test_approximately(a, b):
    allclose = partial(torch.allclose, atol=1e-5, rtol=1e-03)
    if not isinstance(a, torch.Tensor) or not isinstance(b, torch.Tensor):
        a = torch.tensor(a)
        b = torch.tensor(b)
    test(a, b, allclose, "Approximate Equality")

In [15]:
test_equality(1e-5,1e-5)

In [16]:
test_approximately(1e-5, 1e-6)

# Optimizing Matrix Multiplications

**Test Variables**

In [17]:
A = torch.randn([10,10])
B = torch.randn([10,10])

In [18]:
(A@B).shape

torch.Size([10, 10])

## For Loop

In [19]:
def matmul(A,B):
    A_rows, A_cols = A.shape
    B_rows, B_cols = B.shape
    assert A_cols==B_rows,\
    f"Inner dimensions must match: {A_cols} not equal to {B_rows}"
    C = torch.zeros([A_rows, B_cols])
    for i in range(A_rows):
        for j in range(B_cols):
            for k in range(A_cols):
                C[i,j] += A[i,k] * B[k,j]
    return C

In [20]:
matmul(A,B)

tensor([[ 0.4154,  3.3453, -0.5002, -0.9801, -0.4568, -3.1878,  5.1281,  0.4275,
         -1.7929,  0.0387],
        [ 0.2676,  0.1558, -3.1746,  2.8866, -0.1213,  1.3192, -4.8182, -1.1246,
         -0.7001,  1.5069],
        [ 0.9456,  4.1087, -0.2945,  0.8945,  0.0742,  0.4848,  2.1900,  1.8439,
          1.4088, -0.1643],
        [-0.0770, -0.9717,  0.8800,  1.4605,  0.4117,  0.3720, -0.1310,  2.1343,
         -1.2965, -1.2105],
        [-3.8746,  4.5692,  0.6945, -0.7087, -0.4480, -0.0911,  4.5888,  1.2241,
         -2.5403, -1.1710],
        [ 1.7651,  2.2513,  2.3259,  3.3898,  0.7616, -3.7523,  0.6244, -1.8878,
         -3.1584, -1.2781],
        [-0.9508, -0.2111,  1.2930, -0.8729,  0.7032, -1.1108,  0.9099, -2.1492,
         -1.2236,  0.1588],
        [-5.1938,  2.8121, -3.3159,  0.6086, -0.0598,  2.4774, -1.7918, -2.1347,
         -2.1464, -0.4794],
        [-7.6167, -1.5256, -4.3327, -1.7981, -1.5708,  3.5558,  0.6801, -2.5677,
          3.9500,  0.5849],
        [-2.5469,  

In [21]:
test_approximately(matmul(A, B), (A@B))

In [22]:
matmul_loop_time = timeit.timeit(partial(matmul,A,B), number=10)
matmul_loop_time

0.09761166900000262

In [23]:
# Call the matrix multiplication function
C = matmul(A, B)
print("Result Matrix C (A x B):")
print(C)

Result Matrix C (A x B):
tensor([[ 0.4154,  3.3453, -0.5002, -0.9801, -0.4568, -3.1878,  5.1281,  0.4275,
         -1.7929,  0.0387],
        [ 0.2676,  0.1558, -3.1746,  2.8866, -0.1213,  1.3192, -4.8182, -1.1246,
         -0.7001,  1.5069],
        [ 0.9456,  4.1087, -0.2945,  0.8945,  0.0742,  0.4848,  2.1900,  1.8439,
          1.4088, -0.1643],
        [-0.0770, -0.9717,  0.8800,  1.4605,  0.4117,  0.3720, -0.1310,  2.1343,
         -1.2965, -1.2105],
        [-3.8746,  4.5692,  0.6945, -0.7087, -0.4480, -0.0911,  4.5888,  1.2241,
         -2.5403, -1.1710],
        [ 1.7651,  2.2513,  2.3259,  3.3898,  0.7616, -3.7523,  0.6244, -1.8878,
         -3.1584, -1.2781],
        [-0.9508, -0.2111,  1.2930, -0.8729,  0.7032, -1.1108,  0.9099, -2.1492,
         -1.2236,  0.1588],
        [-5.1938,  2.8121, -3.3159,  0.6086, -0.0598,  2.4774, -1.7918, -2.1347,
         -2.1464, -0.4794],
        [-7.6167, -1.5256, -4.3327, -1.7981, -1.5708,  3.5558,  0.6801, -2.5677,
          3.9500,  0.5

In [24]:
matmul_time = timeit.timeit(partial(matmul, A, B), number=1)
print(f"Time taken for matmul(A, B)v6e-1 TPU{matmul_time:.6f} seconds")

Time taken for matmul(A, B)v6e-1 TPU0.010310 seconds
